In [1]:
import numpy as np 
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv("../data/encoded_dataset.csv")
df.head()

,age,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,capital-gains,capital-loss,incident_severity,incident_hour_of_the_day,number_of_vehicles_involved,...,insured_hobbies_hiking,insured_hobbies_kayaking,insured_hobbies_movies,insured_hobbies_paintball,insured_hobbies_polo,insured_hobbies_reading,insured_hobbies_skydiving,insured_hobbies_sleeping,insured_hobbies_video-games,insured_hobbies_yachting
0,48,1000,7.249862,0,466132,53300,0,2,5,1,...,False,False,False,False,False,False,False,True,False,False
1,42,2000,7.088592,5000000,468176,0,0,1,8,1,...,False,False,False,False,False,True,False,False,False,False
2,29,2000,7.254277,5000000,430632,35100,0,1,7,3,...,False,False,False,False,False,False,False,False,False,False
3,41,2000,7.256114,6000000,608117,48900,-62400,2,5,1,...,False,False,False,False,False,False,False,False,False,False
4,44,1000,7.368283,6000000,610706,66000,-46000,1,20,1,...,False,False,False,False,False,False,False,False,False,False


In [3]:
df = df.drop(columns=["policy_number", "incident_id"], errors="ignore")

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

In [5]:
X = df.drop("fraud_reported", axis=1)
y = df["fraud_reported"]

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
X_train = X_train.fillna(0)

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=42))
])

In [10]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 DecisionTreeClassifier(class_weight='balanced',
                                        random_state=42))])

In [11]:
y_pred = pipeline.predict(X_test)

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[126  25]
 [ 22  27]]
              precision    recall  f1-score   support

           0       0.85      0.83      0.84       151
           1       0.52      0.55      0.53        49

    accuracy                           0.77       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.77      0.77      0.77       200



In [13]:
import joblib

joblib.dump(pipeline, "pipeline.pkl")

['pipeline.pkl']

In [14]:
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']

In [15]:
X.columns.tolist()

['age',
 'policy_deductable',
 'policy_annual_premium',
 'umbrella_limit',
 'insured_zip',
 'capital-gains',
 'capital-loss',
 'incident_severity',
 'incident_hour_of_the_day',
 'number_of_vehicles_involved',
 'property_damage',
 'bodily_injuries',
 'witnesses',
 'police_report_available',
 'total_claim_amount',
 'auto_year',
 'insured_sex_MALE',
 'insured_education_level_College',
 'insured_education_level_High School',
 'insured_education_level_JD',
 'insured_education_level_MD',
 'insured_education_level_Masters',
 'insured_education_level_PhD',
 'insured_relationship_not-in-family',
 'insured_relationship_other-relative',
 'insured_relationship_own-child',
 'insured_relationship_unmarried',
 'insured_relationship_wife',
 'policy_state_IN',
 'policy_state_OH',
 'incident_state_NY',
 'incident_state_OH',
 'incident_state_PA',
 'incident_state_SC',
 'incident_state_VA',
 'incident_state_WV',
 'incident_city_Columbus',
 'incident_city_Hillsdale',
 'incident_city_Northbend',
 'incident_

In [16]:
print(y.unique())

[1 0]


In [17]:
print(y_train.value_counts())

fraud_reported
0    602
1    602
Name: count, dtype: int64
